# Первая задача (анализ тональности естественного языка)





In [4]:
# Устанавливаем библиотеку transformers (если еще не установлена)
!pip install transformers torch

from transformers import pipeline

# Создаем пайплайн для анализа тональности
# По умолчанию используется предобученная модель с Hugging Face
sentiment_analyzer = pipeline("sentiment-analysis")

# Тестовые тексты на английском (или русском, если выбрать мультиязычную модель)
# Завуалированные тексты с неочевидным эмоциональным контекстом
texts_en = [
    "I was genuinely surprised to see you actually showed up after everything you said last night.",
    "The book was so gripping that I completely forgot I had a pot of soup burning on the stove.",
    "Thank you for the detailed feedback on my presentation, I didn't know I could make that many mistakes in five minutes."
]

# Прогоняем тексты через модель и выводим результат
results = sentiment_analyzer(texts)
for text, result in zip(texts, results):
    print(f"Текст: {text}")
    print(f"Результат: Метка -> {result['label']}, Уверенность -> {result['score']:.4f}\n")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Текст: I am absolutely thrilled with this course, it's amazing!
Результат: Метка -> POSITIVE, Уверенность -> 0.9999

Текст: This is the worst experience I've ever had. Terrible service.
Результат: Метка -> NEGATIVE, Уверенность -> 0.9997

Текст: The weather today is ok, nothing special.
Результат: Метка -> NEGATIVE, Уверенность -> 0.9961



# Вторая задача (обработка аудио)

In [2]:
# Устанавливаем необходимые библиотеки
!pip install --upgrade pip
!pip install transformers datasets torch soundfile librosa

from transformers import pipeline
import torch

# Загружаем готовый пайплайн для распознавания речи (Automatic Speech Recognition)
# Используем компактную и быструю модель openai/whisper-tiny
asr_pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")

# Для теста можно использовать встроенный короткий аудиофайл из библиотеки datasets
from datasets import load_dataset
dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
sample_audio = dataset[0]["audio"]

# Прогоняем аудио через модель
result = asr_pipeline(sample_audio)
print("Распознанный текст:")
print(result["text"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  151MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

...

clean/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.19MB            

clean/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

Распознанный текст:
 Mr. Quilter is the apostle of the middle classes and we are glad to welcome his gospel.


# Третья задача (обработка изображений)

In [5]:
# Устанавливаем необходимые библиотеки (обычно они уже есть в Colab)
!pip install torch torchvision pillow requests

import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO

# 1. Загружаем предобученную модель ResNet-50
# pretrained=True означает, что мы берем модель, уже обученную на огромном датасете ImageNet
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)
model.eval() # Переводим модель в режим инференса (оценки)

# 2. Скачиваем тестовую картинку (например, котика) из интернета
url = "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=500"
response = requests.get(url)
img = Image.open(BytesIO(response.content)).convert("RGB")

# 3. Подготавливаем картинку для модели (изменяем размер, нормализуем)
preprocess = weights.transforms()
input_tensor = preprocess(img).unsqueeze(0) # Добавляем размерность батча

# 4. Пропускаем картинку через модель
with torch.no_grad():
    output = model(input_tensor)

# 5. Получаем результат (класс с наибольшей вероятностью)
probabilities = torch.nn.functional.softmax(output[0], dim=0)
top5_prob, top5_catid = torch.topk(probabilities, 5)

# Загружаем список меток классов ImageNet для расшифровки ответа
class_names = requests.get("https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt").text.splitlines()

print("Предсказания модели:")
for i in range(5):
    class_name = class_names[top5_catid[i]]
    probability = top5_prob[i].item() * 100
    print(f"{i+1}. {class_name} — {probability:.2f}%")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 326MB/s]


Предсказания модели:
1. Egyptian cat — 33.36%
2. tabby — 5.06%
3. tiger cat — 2.54%
4. paper towel — 2.44%
5. doormat — 1.77%


# Четвёртая задача (обработка видео)

In [10]:
# Устанавливаем необходимые библиотеки
!pip install transformers torch torchvision opencv-python pillow requests

import cv2
from PIL import Image
from transformers import pipeline

# Загружаем надежный пайплайн классификации изображений (будем классифицировать кадры видео)
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

# Создаем простейшее тестовое видео или берем кадр с помощью OpenCV для демонстрации
# Эмулируем кадр: создаем пустую картинку или скачиваем тестовое изображение как кадр видео
import numpy as np
dummy_frame = np.zeros((224, 224, 3), dtype=np.uint8) # черный кадр-заглушка для теста
img = Image.fromarray(dummy_frame)

# Прогоняем «кадр видео» через модель
result = classifier(img)
print("Результат анализа кадра видео:")
for item in result[:3]:
    print(f"- {item['label']}: {item['score']:.4f}")

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Результат анализа кадра видео:
- matchstick: 0.0322
- nematode, nematode worm, roundworm: 0.0283
- bassoon: 0.0130
